# S&P 500 Stock Ranking by Skewness and Kurtosis

Ranks S&P 500 stocks by the shape of their daily log-return distribution over 2024-2025.

**Method**
1. Scrape the current S&P 500 constituents from Wikipedia and download adjusted daily prices from Yahoo Finance.
2. Compute daily log returns for 2024-01-02 to 2025-12-31.
3. Keep stocks with negative skewness and at least 90% data coverage.
4. Score each stock as `0.5 * rank(kurtosis) + 0.5 * rank(-skewness)`, scaled to 0-1. High kurtosis and strongly negative skewness both push a stock up the ranking.

Stocks at the top are those with the heaviest left tails, i.e. returns dominated by a few large single-day drops.


In [ ]:
import pandas as pd
import requests
from io import StringIO

# Download the S&P 500 table from Wikipedia
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text

# Extract the ticker symbols
sp500_tickers = pd.read_html(StringIO(html))[0]["Symbol"].tolist()

# Adapt tickers to Yahoo Finance format (e.g. BRK.B -> BRK-B)
sp500_tickers = [ticker.replace(".", "-") for ticker in sp500_tickers]

sp500_tickers

['MMM',
 'AOS',
 'ABT',
 'ABBV',
 'ACN',
 'ADBE',
 'AMD',
 'AES',
 'AFL',
 'A',
 'APD',
 'ABNB',
 'AKAM',
 'ALB',
 'ARE',
 'ALGN',
 'ALLE',
 'LNT',
 'ALL',
 'GOOGL',
 'GOOG',
 'MO',
 'AMZN',
 'AMCR',
 'AEE',
 'AEP',
 'AXP',
 'AIG',
 'AMT',
 'AWK',
 'AMP',
 'AME',
 'AMGN',
 'APH',
 'ADI',
 'AON',
 'APA',
 'APO',
 'AAPL',
 'AMAT',
 'APP',
 'APTV',
 'ACGL',
 'ADM',
 'ARES',
 'ANET',
 'AJG',
 'AIZ',
 'T',
 'ATO',
 'ADSK',
 'ADP',
 'AZO',
 'AVY',
 'AXON',
 'BKR',
 'BALL',
 'BAC',
 'BAX',
 'BDX',
 'BRK-B',
 'BBY',
 'TECH',
 'BIIB',
 'BLK',
 'BX',
 'XYZ',
 'BE',
 'BNY',
 'BA',
 'BKNG',
 'BSX',
 'BMY',
 'AVGO',
 'BR',
 'BRO',
 'BF-B',
 'BG',
 'BXP',
 'CHRW',
 'CDNS',
 'CPT',
 'COF',
 'CAH',
 'CCL',
 'CARR',
 'CVNA',
 'CASY',
 'CAT',
 'CBOE',
 'CBRE',
 'CDW',
 'COR',
 'CNC',
 'CNP',
 'CF',
 'CRL',
 'SCHW',
 'CHTR',
 'CVX',
 'CMG',
 'CB',
 'CHD',
 'CIEN',
 'CI',
 'CINF',
 'CTAS',
 'CSCO',
 'C',
 'CFG',
 'CLX',
 'CME',
 'CMS',
 'KO',
 'CTSH',
 'COHR',
 'COIN',
 'CL',
 'CMCSA',
 'FIX',
 'COP',
 'E

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf

# 1. Download (end is exclusive, so 2026-01-01 includes 2025-12-31)
prices = yf.download(
    sp500_tickers + ["SPY"],
    start="2023-12-26",      # a few days earlier so the 2024-01-02 return exists
    end="2026-01-01",
    auto_adjust=True,
    group_by="column",
    threads=True,
    progress=True,
)

close = prices["Close"]

[*********************100%***********************]  504 of 504 completed
ERROR:yfinance:
3 Failed downloads:
ERROR:yfinance:['VYLR', 'FDXF', 'HONA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2023-12-26 -> 2026-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1703566800, endDate = 1767243600")')


In [ ]:
prices

Price      Adj Close                 Close                          \
Ticker          FDXF HONA VYLR           A        AAPL        ABBV   
Date                                                                 
2023-12-26       NaN  NaN  NaN  136.724945  190.724777  140.798706   
2023-12-27       NaN  NaN  NaN  136.734756  190.823532  141.035461   
2023-12-28       NaN  NaN  NaN  136.685852  191.248367  140.917084   
2023-12-29       NaN  NaN  NaN  136.192139  190.211029  141.117416   
2024-01-02       NaN  NaN  NaN  135.917831  183.404022  145.533905   
...              ...  ...  ...         ...         ...         ...   
2025-12-24       NaN  NaN  NaN  137.289749  273.066711  224.580185   
2025-12-26       NaN  NaN  NaN  137.359207  272.657867  224.668106   
2025-12-29       NaN  NaN  NaN  136.902634  273.016876  225.508240   
2025-12-30       NaN  NaN  NaN  136.594955  272.338684  224.433655   
2025-12-31       NaN  NaN  NaN  135.056503  271.122009  223.212524   

Price                                                      ...   Volume  \
Ticker            ABNB         ABT       ACGL         ACN  ...       WY   
Date                                                       ...            
2023-12-26  138.720001  103.183754  69.900551  336.017944  ...  2542300   
2023-12-27  136.550003  103.712746  69.929077  336.388733  ...  2109200   
2023-12-28  137.000000  104.288979  70.423546  334.268585  ...  2586100   
2023-12-29  136.139999  103.977249  70.623230  333.622101  ...  3079100   
2024-01-02  134.479996  103.769417  71.840385  329.828674  ...  2317100   
...                ...         ...        ...         ...  ...      ...   
2025-12-24  136.779999  122.532806  96.400002  264.528992  ...  1935800   
2025-12-26  136.820007  122.562256  95.870003  265.616577  ...  3510600   
2025-12-29  136.619995  122.297188  96.389999  265.861511  ...  3841600   
2025-12-30  136.910004  123.485107  96.660004  264.548553  ...  3495800   
2025-12-31  135.720001  123.004051  95.919998  262.882904  ...  4614400   

Price                                                                        \
Ticker         WYNN      XEL       XOM      XYL       XYZ      YUM      ZBH   
Date                                                                          
2023-12-26  1150000  2147500  16835100   440300   6135500   627500  1870700   
2023-12-27  1142200  2865900  14558800  1007700   7418800  1050400  1058600   
2023-12-28  1233000  2421900  16329300   709100   6830500   882500   662200   
2023-12-29  1144800  2813300  17746400   698900   8244900  1196800   849600   
2024-01-02  3013600  4627600  23483000  1089400  15249500  1485300  1087400   
...             ...      ...       ...      ...       ...      ...      ...   
2025-12-24   615000  1427600   6137400   303700   3017700   878600   623200   
2025-12-26   717300  2354300   8066100   484600   3118400  1086400   967500   
2025-12-29  1017200  3999500  14782500   646500   4216700  1963000   924500   
2025-12-30   689900  5420800  11150500   531900   2958800  1158600   934500   
2025-12-31  1949300  2513000  10559900   532000   3487500  1102500  1013900   

Price                        
Ticker        ZBRA      ZTS  
Date                         
2023-12-26  220000   814600  
2023-12-27  275700   766400  
2023-12-28  193900   880100  
2023-12-29  251300  1007200  
2024-01-02  452900  1642300  
...            ...      ...  
2025-12-24  374200  2369000  
2025-12-26  622400  3226800  
2025-12-29  563900  4465600  
2025-12-30  492400  3230200  
2025-12-31  376600  2943300  

[506 rows x 2523 columns]

In [ ]:
log_ret = np.log(close).diff()
log_ret = log_ret.replace([np.inf, -np.inf], np.nan)
log_ret = log_ret.loc["2024-01-02":"2025-12-31"]

In [ ]:
n_sessions = log_ret["SPY"].notna().sum()
log_ret = log_ret.drop(columns="SPY")

In [ ]:
stats = pd.DataFrame({
    "skew": log_ret.skew(),
    "kurtosis": log_ret.kurt(),
    "n_obs": log_ret.count(),
})
stats["coverage"] = stats["n_obs"] / n_sessions

In [ ]:
stats = stats[(stats["skew"] < 0) & (stats["coverage"] >= 0.90)].dropna()

In [ ]:
N = len(stats)
kurt_score = stats["kurtosis"].rank(method="average") / N      # HIGH kurtosis -> high score
skew_score = (-stats["skew"]).rank(method="average") / N

stats["score"] = 0.5 * kurt_score + 0.5 * skew_score

ranking = stats.sort_values("score", ascending=False)
ranking["pct_of_best"] = 10 * ranking["score"] / ranking["score"].iloc[0]
ranking.head(10)

,skew,kurtosis,n_obs,coverage,score,pct_of_best
Ticker,,,,,,
GL,-14.197003,280.999882,502,1.000000,1.000000,10.000000
FISV,-11.872969,196.182838,500,0.996016,0.996997,9.969970
EW,-8.178498,139.148558,502,1.000000,0.993994,9.939940
CNC,-7.933332,124.828369,502,1.000000,0.990991,9.909910
DXCM,-6.839502,101.555263,502,1.000000,0.987988,9.879880
ALGN,-6.125455,91.427183,502,1.000000,0.983483,9.834835
WST,-5.993035,97.250391,502,1.000000,0.981982,9.819820
IT,-6.037778,85.032703,502,1.000000,0.980480,9.804805
SNPS,-5.334732,74.904511,502,1.000000,0.974474,9.744745
